# Train PPO To Mine Ores

In this tutorial you will learn how to run a full PPO training cycle on Factoriax, including visualization of the training process, using JAX.

## Scenario: Mining-v1

**Mining-v1** is a compact benchmark for learning basic navigation and resource collection. You can use it to quickly validate your training setup.

Each episode generates a fresh 8×8 map with ten iron-ore tiles placed at random positions.
Every ore tile holds 3 resources, giving a maximum of 30 collectable items per episode.
The agent earns +1 reward for each ore item it mines manually and has 100 steps to collect as much as possible.

| Property | Value |
|---|---|
| Map size | 8×8 |
| Ore tiles | 10 (random per episode) |
| Resources per tile | 3 |
| Max reward | 30 |
| Episode length | 100 steps |
| Reward signal | +1 per ore item mined |


In [ ]:
from pathlib import Path

import jax
import matplotlib.pyplot as plt
import numpy as np

from factoriax.engine.renderer import JaxRenderer
from factoriax.make import env_from_name

_IMG_DIR = Path("_images")
_IMG_DIR.mkdir(exist_ok=True)

env, params = env_from_name("Mining-v1")
_, state = env.reset_env(jax.random.PRNGKey(0), params)

fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(np.asarray(JaxRenderer(tile_px=32).jit_render_map(state)))
ax.axis("off")
fig.savefig(_IMG_DIR / "mining_initial_state.png", bbox_inches="tight", dpi=120)
plt.close(fig)

![Mining-v1 initial state](_images/mining_initial_state.png)

## The environment interface

Factoriax follows the [gymnax](https://github.com/RobertTLange/gymnax) API. The two core calls are `env.reset_env` and `env.step_env`. Both are **pure functions**. They take all state as arguments and return new state, with no hidden mutation and no global variables. This makes them trivially JIT-compilable and vmappable.

`reset_env(key, params)` returns an observation vector and an `EnvState`. `step_env(key, state, action, params)` returns the next observation, next state, a scalar reward, a done flag, and an info dict.

In [ ]:
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from factoriax.engine.constants import NUM_ACTIONS, Action, BlockType
from factoriax.make import env_from_name

env, params = env_from_name("Mining-v1")

key = jax.random.PRNGKey(0)
obs, state = env.reset_env(key, params)

print("obs shape  :", obs.shape)
print("player pos :", np.array(state.player_positions[0]))
print("ore tiles  :", int(jnp.sum(state.map == int(BlockType.IRON))))
print("resources  :", np.array(state.block_resources))

The reward fires when the player manually mines an ore tile. Let's verify this by navigating to the nearest ore tile and mining it.

In [ ]:
step_fn = jax.jit(env.step_env)

# Locate the nearest ore tile and build an approach path.
map_np = np.array(state.map)
px, py = int(state.player_positions[0, 0]), int(state.player_positions[0, 1])
oy_arr, ox_arr = np.where(map_np == int(BlockType.IRON))

# Find the closest approach position (adjacent to an ore tile).
for ore_x, ore_y in zip(ox_arr, oy_arr):
    valid = [
        (tx, ty, fa)
        for tx, ty, fa in [
            (ore_x - 1, ore_y, Action.FACE_RIGHT),
            (ore_x + 1, ore_y, Action.FACE_LEFT),
            (ore_x, ore_y - 1, Action.FACE_DOWN),
            (ore_x, ore_y + 1, Action.FACE_UP),
        ]
        if 0 <= tx < env.map_width and 0 <= ty < env.map_height
    ]
    if valid:
        target_x, target_y, face = min(
            valid, key=lambda c: abs(c[0] - px) + abs(c[1] - py)
        )
        break

moves = []
cx, cy = px, py
while cx != target_x:
    moves.append(Action.RIGHT if target_x > cx else Action.LEFT)
    cx += 1 if target_x > cx else -1
while cy != target_y:
    moves.append(Action.DOWN if target_y > cy else Action.UP)
    cy += 1 if target_y > cy else -1

# Execute: move, face, mine.
for action in moves + [face, Action.MINE]:
    key, sk = jax.random.split(key)
    obs, state, reward, done, _ = step_fn(sk, state, int(action), params)

print(f"reward after MINE: {float(reward)}")

## Observation and action spaces

The observation is a flat float32 vector encoding the visible map tiles and a set of player scalars. The action space has 85 discrete actions in five families (movement, interaction including `MINE`, placement, crafting, and deposit). For this scenario only the first two families matter.

In [ ]:
from factoriax.engine.constants import (
    CRAFT_BASE,
    DEPOSIT_BASE,
    PLACE_BASE,
    InteractAction,
    MoveAction,
)

print("Observation shape :", env.observation_space(params).shape)
print("Action count      :", env.action_space(params).n)
print()
print(
    "Move actions    (0–{})  : {}".format(
        len(MoveAction) - 1, [a.name for a in MoveAction]
    )
)
print(
    "Interact actions({}-{}) : {}".format(
        len(MoveAction), PLACE_BASE - 1, [a.name for a in InteractAction]
    )
)

## Efficient rollouts

Running many environments in parallel is the key to fast RL training in JAX. `jax.vmap` maps a function over a batch of independent keys, and `jax.lax.scan` replaces the Python step loop with a single compiled XLA kernel. Together they let one `jax.jit` call collect thousands of episodes without returning to Python.

See the [Getting Started guide](../start-here/getting_started.ipynb) for a step-by-step build-up of this pattern. Here we go straight to the most efficient form and use it to establish a random-agent baseline.

In [ ]:
def rollout(key):
    def scan_body(carry, _):
        key, state = carry
        key, act_key, step_key = jax.random.split(key, 3)
        action = jax.random.randint(act_key, (), 0, NUM_ACTIONS)
        obs, state, reward, done, _ = env.step_env(step_key, state, action, params)
        return (key, state), reward

    obs, state = env.reset_env(key, params)
    (_, _), rewards = jax.lax.scan(
        scan_body, (key, state), None, length=params.max_timesteps
    )
    return rewards.sum()


collect = jax.jit(jax.vmap(rollout))

N = 512
keys = jax.random.split(jax.random.PRNGKey(1), N)
random_returns = np.asarray(collect(keys))
print(
    f"Random agent over {N} episodes: mean return = {random_returns.mean():.2f} ± {random_returns.std():.2f}"
)

## Training with PureJaxRL

[PureJaxRL](https://github.com/luchris429/purejaxrl) provides a fully end-to-end JAX PPO implementation. The entire training loop (rollout collection, GAE, minibatch updates) is compiled into a single XLA program via `jax.lax.scan` and `jax.vmap`, so Python is called only once regardless of how many updates you run.

The code below is copied verbatim from PureJaxRL, with one change. Instead of creating the environment via `gymnax.make`, we pass our factoriax env and params in through the config dict.

In [ ]:
# ── From PureJaxRL/wrappers.py ────────────────────────────────────────────
from typing import Any, NamedTuple, Optional, Sequence, Tuple, Union

import chex
from flax import struct
from gymnax.environments import environment, spaces


class GymnaxWrapper(object):
    """Base class for Gymnax wrappers."""

    def __init__(self, env):
        self._env = env

    def __getattr__(self, name):
        return getattr(self._env, name)


@struct.dataclass
class LogEnvState:
    env_state: environment.EnvState
    episode_returns: float
    episode_lengths: int
    returned_episode_returns: float
    returned_episode_lengths: int
    timestep: int


class LogWrapper(GymnaxWrapper):
    """Log the episode returns and lengths."""

    def __init__(self, env: environment.Environment):
        super().__init__(env)

    def reset(
        self, key: chex.PRNGKey, params: Optional[environment.EnvParams] = None
    ) -> Tuple[chex.Array, environment.EnvState]:
        obs, env_state = self._env.reset_env(key, params)
        state = LogEnvState(env_state, 0, 0, 0, 0, 0)
        return obs, state

    def step(
        self,
        key: chex.PRNGKey,
        state: LogEnvState,
        action: Union[int, float],
        params: Optional[environment.EnvParams] = None,
    ) -> Tuple[chex.Array, environment.EnvState, float, bool, dict]:
        obs, env_state, reward, done, info = self._env.step_env(
            key, state.env_state, action, params
        )
        new_episode_return = state.episode_returns + reward
        new_episode_length = state.episode_lengths + 1
        log_state = LogEnvState(
            env_state=env_state,
            episode_returns=new_episode_return * (1 - done),
            episode_lengths=new_episode_length * (1 - done),
            returned_episode_returns=state.returned_episode_returns * (1 - done)
            + new_episode_return * done,
            returned_episode_lengths=state.returned_episode_lengths * (1 - done)
            + new_episode_length * done,
            timestep=state.timestep + 1,
        )
        info["returned_episode_returns"] = log_state.returned_episode_returns
        info["returned_episode_lengths"] = log_state.returned_episode_lengths
        info["timestep"] = state.timestep
        info["returned_episode"] = done
        return obs, log_state, reward, done, info


# ── From PureJaxRL/ppo.py ─────────────────────────────────────────────────
import distrax
import flax.linen as nn
import optax
from flax.linen.initializers import constant, orthogonal
from flax.training.train_state import TrainState


class ActorCritic(nn.Module):
    action_dim: Sequence[int]
    activation: str = "tanh"

    @nn.compact
    def __call__(self, x):
        if self.activation == "relu":
            activation = nn.relu
        else:
            activation = nn.tanh
        actor_mean = nn.Dense(
            64, kernel_init=orthogonal(np.sqrt(2)), bias_init=constant(0.0)
        )(x)
        actor_mean = activation(actor_mean)
        actor_mean = nn.Dense(
            64, kernel_init=orthogonal(np.sqrt(2)), bias_init=constant(0.0)
        )(actor_mean)
        actor_mean = activation(actor_mean)
        actor_mean = nn.Dense(
            self.action_dim, kernel_init=orthogonal(0.01), bias_init=constant(0.0)
        )(actor_mean)
        pi = distrax.Categorical(logits=actor_mean)

        critic = nn.Dense(
            64, kernel_init=orthogonal(np.sqrt(2)), bias_init=constant(0.0)
        )(x)
        critic = activation(critic)
        critic = nn.Dense(
            64, kernel_init=orthogonal(np.sqrt(2)), bias_init=constant(0.0)
        )(critic)
        critic = activation(critic)
        critic = nn.Dense(1, kernel_init=orthogonal(1.0), bias_init=constant(0.0))(
            critic
        )

        return pi, jnp.squeeze(critic, axis=-1)


class Transition(NamedTuple):
    done: jnp.ndarray
    action: jnp.ndarray
    value: jnp.ndarray
    reward: jnp.ndarray
    log_prob: jnp.ndarray
    obs: jnp.ndarray
    info: jnp.ndarray


def make_train(config):
    config["NUM_UPDATES"] = (
        config["TOTAL_TIMESTEPS"] // config["NUM_STEPS"] // config["NUM_ENVS"]
    )
    config["MINIBATCH_SIZE"] = (
        config["NUM_ENVS"] * config["NUM_STEPS"] // config["NUM_MINIBATCHES"]
    )
    # ── Only change from upstream: accept env + params from config ──────────
    env, env_params = config["ENV"], config["ENV_PARAMS"]
    env = LogWrapper(env)

    def linear_schedule(count):
        frac = (
            1.0
            - (count // (config["NUM_MINIBATCHES"] * config["UPDATE_EPOCHS"]))
            / config["NUM_UPDATES"]
        )
        return config["LR"] * frac

    def train(rng):
        # INIT NETWORK
        network = ActorCritic(
            env.action_space(env_params).n, activation=config["ACTIVATION"]
        )
        rng, _rng = jax.random.split(rng)
        init_x = jnp.zeros(env.observation_space(env_params).shape)
        network_params = network.init(_rng, init_x)
        if config["ANNEAL_LR"]:
            tx = optax.chain(
                optax.clip_by_global_norm(config["MAX_GRAD_NORM"]),
                optax.adam(learning_rate=linear_schedule, eps=1e-5),
            )
        else:
            tx = optax.chain(
                optax.clip_by_global_norm(config["MAX_GRAD_NORM"]),
                optax.adam(config["LR"], eps=1e-5),
            )
        train_state = TrainState.create(
            apply_fn=network.apply,
            params=network_params,
            tx=tx,
        )

        # INIT ENV
        rng, _rng = jax.random.split(rng)
        reset_rng = jax.random.split(_rng, config["NUM_ENVS"])
        obsv, env_state = jax.vmap(env.reset, in_axes=(0, None))(reset_rng, env_params)

        # TRAIN LOOP
        def _update_step(runner_state, unused):
            # COLLECT TRAJECTORIES
            def _env_step(runner_state, unused):
                train_state, env_state, last_obs, rng = runner_state

                # SELECT ACTION
                rng, _rng = jax.random.split(rng)
                pi, value = network.apply(train_state.params, last_obs)
                action = pi.sample(seed=_rng)
                log_prob = pi.log_prob(action)

                # STEP ENV
                rng, _rng = jax.random.split(rng)
                rng_step = jax.random.split(_rng, config["NUM_ENVS"])
                obsv, env_state, reward, done, info = jax.vmap(
                    env.step, in_axes=(0, 0, 0, None)
                )(rng_step, env_state, action, env_params)
                transition = Transition(
                    done, action, value, reward, log_prob, last_obs, info
                )
                runner_state = (train_state, env_state, obsv, rng)
                return runner_state, transition

            runner_state, traj_batch = jax.lax.scan(
                _env_step, runner_state, None, config["NUM_STEPS"]
            )

            # CALCULATE ADVANTAGE
            train_state, env_state, last_obs, rng = runner_state
            _, last_val = network.apply(train_state.params, last_obs)

            def _calculate_gae(traj_batch, last_val):
                def _get_advantages(gae_and_next_value, transition):
                    gae, next_value = gae_and_next_value
                    done, value, reward = (
                        transition.done,
                        transition.value,
                        transition.reward,
                    )
                    delta = reward + config["GAMMA"] * next_value * (1 - done) - value
                    gae = (
                        delta
                        + config["GAMMA"] * config["GAE_LAMBDA"] * (1 - done) * gae
                    )
                    return (gae, value), gae

                _, advantages = jax.lax.scan(
                    _get_advantages,
                    (jnp.zeros_like(last_val), last_val),
                    traj_batch,
                    reverse=True,
                    unroll=16,
                )
                return advantages, advantages + traj_batch.value

            advantages, targets = _calculate_gae(traj_batch, last_val)

            # UPDATE NETWORK
            def _update_epoch(update_state, unused):
                def _update_minbatch(train_state, batch_info):
                    traj_batch, advantages, targets = batch_info

                    def _loss_fn(params, traj_batch, gae, targets):
                        # RERUN NETWORK
                        pi, value = network.apply(params, traj_batch.obs)
                        log_prob = pi.log_prob(traj_batch.action)

                        # CALCULATE VALUE LOSS
                        value_pred_clipped = traj_batch.value + (
                            value - traj_batch.value
                        ).clip(-config["CLIP_EPS"], config["CLIP_EPS"])
                        value_losses = jnp.square(value - targets)
                        value_losses_clipped = jnp.square(value_pred_clipped - targets)
                        value_loss = (
                            0.5 * jnp.maximum(value_losses, value_losses_clipped).mean()
                        )

                        # CALCULATE ACTOR LOSS
                        ratio = jnp.exp(log_prob - traj_batch.log_prob)
                        gae = (gae - gae.mean()) / (gae.std() + 1e-8)
                        loss_actor1 = ratio * gae
                        loss_actor2 = (
                            jnp.clip(
                                ratio,
                                1.0 - config["CLIP_EPS"],
                                1.0 + config["CLIP_EPS"],
                            )
                            * gae
                        )
                        loss_actor = -jnp.minimum(loss_actor1, loss_actor2)
                        loss_actor = loss_actor.mean()
                        entropy = pi.entropy().mean()

                        total_loss = (
                            loss_actor
                            + config["VF_COEF"] * value_loss
                            - config["ENT_COEF"] * entropy
                        )
                        return total_loss, (value_loss, loss_actor, entropy)

                    grad_fn = jax.value_and_grad(_loss_fn, has_aux=True)
                    total_loss, grads = grad_fn(
                        train_state.params, traj_batch, advantages, targets
                    )
                    train_state = train_state.apply_gradients(grads=grads)
                    return train_state, total_loss

                train_state, traj_batch, advantages, targets, rng = update_state
                rng, _rng = jax.random.split(rng)
                batch_size = config["MINIBATCH_SIZE"] * config["NUM_MINIBATCHES"]
                assert batch_size == config["NUM_STEPS"] * config["NUM_ENVS"], (
                    "batch size must be equal to number of steps * number of envs"
                )
                permutation = jax.random.permutation(_rng, batch_size)
                batch = (traj_batch, advantages, targets)
                batch = jax.tree_util.tree_map(
                    lambda x: x.reshape((batch_size,) + x.shape[2:]), batch
                )
                shuffled_batch = jax.tree_util.tree_map(
                    lambda x: jnp.take(x, permutation, axis=0), batch
                )
                minibatches = jax.tree_util.tree_map(
                    lambda x: jnp.reshape(
                        x, [config["NUM_MINIBATCHES"], -1] + list(x.shape[1:])
                    ),
                    shuffled_batch,
                )
                train_state, total_loss = jax.lax.scan(
                    _update_minbatch, train_state, minibatches
                )
                update_state = (train_state, traj_batch, advantages, targets, rng)
                return update_state, total_loss

            update_state = (train_state, traj_batch, advantages, targets, rng)
            update_state, loss_info = jax.lax.scan(
                _update_epoch, update_state, None, config["UPDATE_EPOCHS"]
            )
            train_state = update_state[0]
            metric = traj_batch.info
            rng = update_state[-1]

            runner_state = (train_state, env_state, last_obs, rng)
            return runner_state, metric

        rng, _rng = jax.random.split(rng)
        runner_state = (train_state, env_state, obsv, _rng)
        runner_state, metric = jax.lax.scan(
            _update_step, runner_state, None, config["NUM_UPDATES"]
        )
        return {"runner_state": runner_state, "metrics": metric}

    return train

In [ ]:
config = {
    "LR": 2.5e-4,
    "NUM_ENVS": 64,
    "NUM_STEPS": params.max_timesteps,  # one full episode per rollout (100 steps)
    "TOTAL_TIMESTEPS": 1_000_000,
    "UPDATE_EPOCHS": 4,
    "NUM_MINIBATCHES": 4,
    "GAMMA": 0.99,
    "GAE_LAMBDA": 0.95,
    "CLIP_EPS": 0.2,
    "ENT_COEF": 0.01,
    "VF_COEF": 0.5,
    "MAX_GRAD_NORM": 0.5,
    "ACTIVATION": "tanh",
    "ANNEAL_LR": True,
    # ── factoriax env (the only change from upstream PureJaxRL) ──────────
    "ENV": env,
    "ENV_PARAMS": params,
}

train_fn = jax.jit(make_train(config))
out = train_fn(jax.random.PRNGKey(42))
print("Training complete.")

In [ ]:
import gc

# Extract what we need before releasing the full training output.
trained_params = out["runner_state"][0].params
metrics = out["metrics"]

del out, train_fn
gc.collect()
jax.clear_caches()
print("GPU memory released.")

## Results

`out["metrics"]` contains the info dict logged by `LogWrapper` at every step of every update. `returned_episode_returns` holds the final return of each episode (zero on steps where no episode ended), and `returned_episode` is the boolean mask that tells us which entries are real.

In [ ]:
ep_returns = metrics["returned_episode_returns"]  # (num_updates, num_steps, num_envs)
ep_done = metrics["returned_episode"]  # same shape, bool mask

# Mean return of completed episodes per update iteration.
mean_per_update = np.array(
    [
        float(ep_returns[i][ep_done[i]].mean()) if ep_done[i].any() else float("nan")
        for i in range(ep_returns.shape[0])
    ]
)

_IMG_DIR = Path("_images")
_IMG_DIR.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 3))
updates = np.arange(len(mean_per_update))
ax.plot(updates, mean_per_update, lw=1.5, label="PPO")
ax.axhline(
    random_returns.mean(),
    color="gray",
    ls="--",
    lw=1,
    label=f"random ({random_returns.mean():.1f})",
)
ax.set_xlabel("update")
ax.set_ylabel("mean episode return")
ax.set_title("Mining-v1 PPO learning curve")
ax.legend()
fig.tight_layout()
fig.savefig(_IMG_DIR / "ppo_learning_curve.png", dpi=120)
plt.close(fig)

![PPO learning curve](_images/ppo_learning_curve.png)

In [ ]:
from factoriax.engine.renderer import JaxRenderer

network = ActorCritic(env.action_space(params).n, activation=config["ACTIVATION"])
apply_fn = jax.jit(network.apply)

# Run one greedy episode with the trained policy.
key = jax.random.PRNGKey(99)
obs, state = env.reset_env(key, params)
trained_return = 0.0
for _ in range(params.max_timesteps):
    pi, _ = apply_fn(trained_params, obs)
    action = pi.mode()
    key, sk = jax.random.split(key)
    obs, state, reward, done, _ = step_fn(sk, state, int(action), params)
    trained_return += float(reward)
    if done:
        break

print(f"Random agent : {random_returns.mean():.2f} mean return")
print(f"Trained agent: {trained_return:.2f} return (greedy, single episode)")
print(f"Max possible : {30.0:.2f}")

fig, ax = plt.subplots(figsize=(3, 3))
renderer = JaxRenderer(tile_px=32)
ax.imshow(np.asarray(renderer.jit_render_map(state)))
ax.axis("off")
ax.set_title("End state (trained agent)")
fig.tight_layout()
fig.savefig(_IMG_DIR / "ppo_trained_end_state.png", dpi=120)
plt.close(fig)

![Trained agent end state](_images/ppo_trained_end_state.png)

## The Training Setup

For this tutorial we will be using a pre-built JAX rl library called [PureJaxRL](https://github.com/luchris429/purejaxrl). We will be covering

- Options for the observation and action spaces
- Wrappers and how they work
- Training and testing visualization, including trajectory visualizations using the analysis module
- Saving and loading models